## Dataset - Porto Seguro Safe Driver Dataset

In [1]:
import pandas as pd
import numpy as np

df_train = pd.read_csv('../Cleaned-Dataset/train.csv')
df_test = pd.read_csv('../Cleaned-Dataset/test.csv')

In [2]:
def dataset_split(df_train, df_test):
    train_samples = int(df_train.shape[0] * 0.8)

    X_train, y_train = df_train[:train_samples].drop(columns=['target']), df_train[:train_samples]['target']
    X_valid, y_valid = df_train[train_samples:].drop(columns=['target']), df_train[train_samples:]['target']

    test = df_test
    return X_train, y_train, X_valid, y_valid, test

In [3]:
X_train, y_train, X_valid, y_valid, test = dataset_split(df_train, df_test)

## Light GBM Tree

In [14]:
import numpy as np
from sklearn.metrics import roc_auc_score

class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None,
                 leaf_weight=None, leaf_count=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.leaf_weight = leaf_weight
        self.leaf_count = leaf_count


class DecisionTreeLightGBMStyle:
    """
    FULLY VECTORIZED LightGBM-style tree with histogram binning.
    • Histogram binning (max_bin) – O(max_bin) per feature
    • Vectorized gain computation via np.cumsum + boolean masking (no Python for-loops over bins)
    • Leaf-wise (best-first) growth
    • All original parameters preserved + max_bin
    This is now the fastest possible pure-numpy version while staying faithful to your XGBoost-style API.
    """
    def __init__(self,
                 num_leaves=31,
                 max_depth=-1,
                 min_data_in_leaf=20,
                 min_sum_hessian_in_leaf=1e-3,
                 feature_fraction=0.8,
                 bagging_fraction=0.8,
                 reg_alpha=0.0,
                 reg_lambda=1.0,
                 min_split_gain=0.0,
                 max_bin=255,          # histogram resolution (core LightGBM param)
                 seed=42):
        self.num_leaves = num_leaves
        self.max_depth = max_depth
        self.min_data_in_leaf = min_data_in_leaf
        self.min_sum_hessian_in_leaf = min_sum_hessian_in_leaf
        self.feature_fraction = feature_fraction
        self.bagging_fraction = bagging_fraction
        self.reg_alpha = reg_alpha
        self.reg_lambda = reg_lambda
        self.min_split_gain = min_split_gain
        self.max_bin = max_bin
        self.seed = seed
        self.root = None
        np.random.seed(seed)

        # Set once in fit()
        self.X = None
        self.bin_edges = None
        self.bin_ids = None

    def _compute_bin_edges(self, X):
        """Pre-compute histogram bin edges once per tree (global on the data passed to this tree)."""
        bin_edges = []
        for f in range(X.shape[1]):
            col = X[:, f]
            unique_vals = np.unique(col)
            if len(unique_vals) <= self.max_bin:          # Porto _cat columns (low cardinality)
                edges = np.sort(unique_vals)
            else:                                         # continuous features
                edges = np.percentile(col, np.linspace(0, 100, self.max_bin + 1))
                edges = np.unique(edges)                  # remove duplicate edges
            bin_edges.append(edges)
        return bin_edges

    def _optimal_leaf_weight(self, G, H):
        w = -G / (H + self.reg_lambda)
        if self.reg_alpha > 0:
            w = -np.sign(G) * max(0.0, abs(G) - self.reg_alpha) / (H + self.reg_lambda)
        return w

    def _info_gain_vectorized(self, G, H, G_L, H_L, G_R, H_R):
        """Fully vectorized gain (used after cumsum)."""
        left_gain = (G_L ** 2) / (H_L + self.reg_lambda)
        right_gain = (G_R ** 2) / (H_R + self.reg_lambda)
        parent_gain = (G ** 2) / (H + self.reg_lambda)
        return 0.5 * (left_gain + right_gain - parent_gain) - self.min_split_gain

    def _best_split(self, grad, hess, leaf_indices):
        """Histogram + FULLY VECTORIZED split finding (no Python loops over bins)."""
        m = len(leaf_indices)
        if m < 2 * self.min_data_in_leaf:
            return None, None, None

        n_features = self.X.shape[1]
        features = np.arange(n_features)
        if self.feature_fraction < 1.0:
            n_select = max(1, int(self.feature_fraction * n_features))
            features = np.random.choice(n_features, n_select, replace=False)

        best_gain = -np.inf
        best_feature = best_threshold = None

        G = np.sum(grad[leaf_indices])
        H = np.sum(hess[leaf_indices])

        for f in features:
            bin_ids_f = self.bin_ids[leaf_indices, f]
            num_bins = len(self.bin_edges[f])

            # Build histograms – fully vectorized (C-speed)
            g_hist = np.bincount(bin_ids_f, weights=grad[leaf_indices], minlength=num_bins)
            h_hist = np.bincount(bin_ids_f, weights=hess[leaf_indices], minlength=num_bins)
            count_hist = np.bincount(bin_ids_f, minlength=num_bins)

            # VECTORIZED cumulative sums (eliminates the old Python for-loop)
            g_cum = np.cumsum(g_hist)
            h_cum = np.cumsum(h_hist)
            count_cum = np.cumsum(count_hist)

            G_L = g_cum[:-1]
            H_L = h_cum[:-1]
            left_count = count_cum[:-1]
            G_R = G - G_L
            H_R = H - H_L
            right_count = m - left_count

            # Vectorized validity mask
            valid = ((left_count >= self.min_data_in_leaf) &
                     (right_count >= self.min_data_in_leaf) &
                     (H_L >= self.min_sum_hessian_in_leaf) &
                     (H_R >= self.min_sum_hessian_in_leaf))

            if not np.any(valid):
                continue

            # Vectorized gain computation
            gains = self._info_gain_vectorized(G, H, G_L, H_L, G_R, H_R)
            gains[~valid] = -np.inf

            # Best bin for this feature
            best_b = np.argmax(gains)
            gain = gains[best_b]

            if gain > best_gain:
                best_gain = gain
                best_feature = f
                best_threshold = self.bin_edges[f][best_b + 1]   # split at the right edge of the bin

        if best_gain < self.min_split_gain:
            return None, None, None
        return best_feature, best_threshold, best_gain

    def fit(self, X, grad, hess):
        """Leaf-wise growth + histogram + fully vectorized splits."""
        X = np.array(X, dtype=np.float64)
        grad = np.array(grad, dtype=np.float64)
        hess = np.array(hess, dtype=np.float64)

        self.X = X
        n_samples = X.shape[0]

        indices = np.arange(n_samples)
        if self.bagging_fraction < 1.0:
            sample_size = int(self.bagging_fraction * n_samples)
            indices = np.random.choice(indices, sample_size, replace=False)

        # Histogram preprocessing (once per tree)
        self.bin_edges = self._compute_bin_edges(X)
        self.bin_ids = np.empty((n_samples, X.shape[1]), dtype=np.int32)
        for f in range(X.shape[1]):
            self.bin_ids[:, f] = np.searchsorted(self.bin_edges[f], X[:, f])
            self.bin_ids[:, f] = np.minimum(self.bin_ids[:, f], len(self.bin_edges[f]) - 1)

        # Root leaf
        G = np.sum(grad[indices])
        H = np.sum(hess[indices])
        w = self._optimal_leaf_weight(G, H)
        self.root = Node(leaf_weight=w, leaf_count=len(indices))

        active_leaves = [(self.root, indices.copy())]
        current_num_leaves = 1

        while current_num_leaves < self.num_leaves:
            best_gain = -np.inf
            best_leaf_idx = -1
            best_feature = best_threshold = None
            best_left_indices = best_right_indices = None

            for i, (leaf_node, leaf_indices) in enumerate(active_leaves):
                if len(leaf_indices) < 2 * self.min_data_in_leaf:
                    continue
                feat, thresh, gain = self._best_split(grad, hess, leaf_indices)
                if feat is None:
                    continue
                if gain > best_gain:
                    best_gain = gain
                    best_leaf_idx = i
                    best_feature = feat
                    best_threshold = thresh
                    mask = self.X[leaf_indices, feat] <= thresh
                    best_left_indices = leaf_indices[mask]
                    best_right_indices = leaf_indices[~mask]

            if best_leaf_idx == -1 or best_gain <= 0:
                break

            # Split
            best_leaf_node, _ = active_leaves[best_leaf_idx]
            best_leaf_node.feature = best_feature
            best_leaf_node.threshold = best_threshold
            best_leaf_node.leaf_weight = None

            G_L = np.sum(grad[best_left_indices])
            H_L = np.sum(hess[best_left_indices])
            w_L = self._optimal_leaf_weight(G_L, H_L)
            left_node = Node(leaf_weight=w_L, leaf_count=len(best_left_indices))

            G_R = np.sum(grad[best_right_indices])
            H_R = np.sum(hess[best_right_indices])
            w_R = self._optimal_leaf_weight(G_R, H_R)
            right_node = Node(leaf_weight=w_R, leaf_count=len(best_right_indices))

            best_leaf_node.left = left_node
            best_leaf_node.right = right_node

            active_leaves.pop(best_leaf_idx)
            active_leaves.append((left_node, best_left_indices))
            active_leaves.append((right_node, best_right_indices))
            current_num_leaves += 1

        return self

    def _predict_one(self, x, node):
        if node.leaf_weight is not None:
            return node.leaf_weight
        if x[node.feature] <= node.threshold:
            return self._predict_one(x, node.left)
        return self._predict_one(x, node.right)

    def predict(self, X):
        X = np.array(X, dtype=np.float64)
        return np.array([self._predict_one(x, self.root) for x in X])

## LightGBM

In [ ]:
class LightGBM:
    """
    Complete vectorized LightGBM from scratch – ready for Porto Seguro.
    • Histogram binning (max_bin)
    • Fully vectorized split finding (np.cumsum + masking)
    • Leaf-wise growth + GOSS
    • All important parameters exposed
    Expected speed on 595k × 57 Porto data (typical laptop):
      • First tree: 5–30 seconds
      • 300 trees with GOSS: 30–90 minutes total (much faster than before)
    """
    def __init__(self,
                 num_iterations=100,
                 learning_rate=0.05,
                 num_leaves=31,
                 min_data_in_leaf=20,
                 min_sum_hessian_in_leaf=1e-3,
                 feature_fraction=0.8,
                 bagging_fraction=0.8,
                 reg_alpha=0.0,
                 reg_lambda=1.0,
                 min_split_gain=0.0,
                 max_bin=255,                    # new – controls histogram resolution
                 boosting_type='goss',           # 'goss' strongly recommended for speed
                 top_rate=0.2,
                 other_rate=0.1,
                 scale_pos_weight=1.0,
                 seed=42):
        self.num_iterations = num_iterations
        self.learning_rate = learning_rate
        self.num_leaves = num_leaves
        self.min_data_in_leaf = min_data_in_leaf
        self.min_sum_hessian_in_leaf = min_sum_hessian_in_leaf
        self.feature_fraction = feature_fraction
        self.bagging_fraction = bagging_fraction
        self.reg_alpha = reg_alpha
        self.reg_lambda = reg_lambda
        self.min_split_gain = min_split_gain
        self.max_bin = max_bin
        self.boosting_type = boosting_type
        self.top_rate = top_rate
        self.other_rate = other_rate
        self.scale_pos_weight = scale_pos_weight
        self.seed = seed

        self.trees = []
        self.base_score = None
        np.random.seed(seed)

    def _apply_goss(self, grad, hess):
        if self.boosting_type != 'goss':
            return np.arange(len(grad))
        n = len(grad)
        abs_grad = np.abs(grad)
        sorted_idx = np.argsort(abs_grad)[::-1]
        n_top = int(self.top_rate * n)
        n_other = int(self.other_rate * n)
        top_indices = sorted_idx[:n_top]
        other_candidates = sorted_idx[n_top:]
        other_sample = np.random.choice(other_candidates, n_other, replace=False) if n_other > 0 and len(other_candidates) > 0 else np.array([], dtype=int)
        sampled_indices = np.concatenate([top_indices, other_sample])
        sampled_grad = grad[sampled_indices].copy()
        sampled_hess = hess[sampled_indices].copy()
        adjust_factor = (1.0 - self.top_rate) / self.other_rate if self.other_rate > 0 else 1.0
        small_mask = ~np.isin(sampled_indices, top_indices)
        sampled_grad[small_mask] *= adjust_factor
        sampled_hess[small_mask] *= adjust_factor
        return sampled_indices, sampled_grad, sampled_hess

    def fit(self, X, y):
        X = np.array(X, dtype=np.float64)
        y = np.array(y).ravel().astype(np.float64)

        n_pos = np.sum(y == 1)
        n_neg = len(y) - n_pos
        if n_pos == 0 or n_neg == 0:
            self.base_score = 0.0
        else:
            p_eff = (self.scale_pos_weight * n_pos) / (self.scale_pos_weight * n_pos + n_neg)
            self.base_score = np.log(p_eff / (1 - p_eff)) if 0 < p_eff < 1 else 0.0

        print(f"LightGBM from Scratch (vectorized) -> base_score = {self.base_score:.6f} "
              f"(initial prob = {1/(1+np.exp(-self.base_score)):.6f}, "
              f"scale_pos_weight={self.scale_pos_weight})")

        pred = np.full(len(y), self.base_score, dtype=np.float64)

        for t in range(self.num_iterations):
            p = 1 / (1 + np.exp(-pred))
            grad = p - y
            hess = p * (1 - p)
            grad[y == 1] *= self.scale_pos_weight
            hess[y == 1] *= self.scale_pos_weight

            if self.boosting_type == 'goss':
                sampled_idx, grad_sample, hess_sample = self._apply_goss(grad, hess)
                X_for_tree = X[sampled_idx]
            else:
                sampled_idx = np.arange(len(grad))
                grad_sample = grad
                hess_sample = hess
                X_for_tree = X

            # Build tree (now fully vectorized)
            tree = DecisionTreeLightGBMStyle(
                num_leaves=self.num_leaves,
                min_data_in_leaf=self.min_data_in_leaf,
                min_sum_hessian_in_leaf=self.min_sum_hessian_in_leaf,
                feature_fraction=self.feature_fraction,
                bagging_fraction=self.bagging_fraction,
                reg_alpha=self.reg_alpha,
                reg_lambda=self.reg_lambda,
                min_split_gain=self.min_split_gain,
                max_bin=self.max_bin,
                seed=self.seed + t
            )
            tree.fit(X_for_tree, grad_sample, hess_sample)

            tree_output = tree.predict(X)          # predict on full data
            pred += self.learning_rate * tree_output
            self.trees.append(tree)

            if (t + 1) % 20 == 0 or t == 0:
                current_p = 1 / (1 + np.exp(-pred))
                train_auc = roc_auc_score(y_true=y, y_score=current_p)
                print(f"Tree {t+1}/{self.num_iterations} | Train AUC: {train_auc:.5f} "
                      f"({'GOSS' if self.boosting_type=='goss' else 'GBDT'})")

        return self

    def predict_raw(self, X):
        X = np.array(X, dtype=np.float64)
        raw = np.full(X.shape[0], self.base_score, dtype=np.float64)
        for tree in self.trees:
            raw += self.learning_rate * tree.predict(X)
        return raw

    def predict(self, X, threshold=0.5):
        raw = self.predict_raw(X)
        return (raw >= threshold).astype(int)

    def predict_proba(self, X):
        raw = self.predict_raw(X)
        return 1 / (1 + np.exp(-raw))

## Model training

In [16]:
model = LightGBM(
    num_iterations=300,          # start here (increase later)
    learning_rate=0.05,
    num_leaves=31,               # keep small for speed in scratch version
    min_data_in_leaf=20,
    feature_fraction=0.8,
    bagging_fraction=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    max_bin=255,
    boosting_type='goss',        # critical for speed on 595k rows
    top_rate=0.2,
    other_rate=0.1,
    scale_pos_weight=27.8,       # Porto imbalance
    seed=42
)
model.fit(X_train, y_train)

val_proba = model.predict_proba(X_valid)
print("Val AUC:", roc_auc_score(y_valid, val_proba))

LightGBM from Scratch (vectorized) -> base_score = 0.049434 (initial prob = 0.512356, scale_pos_weight=27.8)
Tree 1/300 | Train AUC: 0.57494 (GOSS)
Tree 20/300 | Train AUC: 0.59600 (GOSS)
Tree 40/300 | Train AUC: 0.59696 (GOSS)
Tree 60/300 | Train AUC: 0.59727 (GOSS)
Tree 80/300 | Train AUC: 0.59727 (GOSS)
Tree 100/300 | Train AUC: 0.59727 (GOSS)
Tree 120/300 | Train AUC: 0.59727 (GOSS)
Tree 140/300 | Train AUC: 0.59761 (GOSS)
Tree 160/300 | Train AUC: 0.59761 (GOSS)
Tree 180/300 | Train AUC: 0.59841 (GOSS)
Tree 200/300 | Train AUC: 0.59841 (GOSS)
Tree 220/300 | Train AUC: 0.59841 (GOSS)
Tree 240/300 | Train AUC: 0.59841 (GOSS)


MemoryError: Unable to allocate 196. MiB for an array with shape (476169, 54) and data type float64